# 03. PPO 학습 실험 히스토리 분석

**목적**: exp001~exp005까지의 학습 과정과 핵심 문제를 분석한다.

**핵심 질문**:
- 각 실험에서 무엇이 달랐는가?
- 왜 모든 실험에서 후반에 0 거래로 수렴하는가?
- exp006에서 무엇을 바꿔야 하는가?

**실험 흐름**:
```
exp001 (기본) → exp002 (lr 튜닝) → exp003 (VecNorm+cosine, 버그) → exp004 (버그수정+ATR) → exp005 (최소 gap 수정)
```

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import os
os.makedirs('../reports/semester1/figures', exist_ok=True)
print('라이브러리 로드 완료')

## 1. 실험 히스토리 요약

각 실험의 변경사항, best Sharpe, 문제점을 표로 정리한다.

In [ ]:
history = [
    {
        'ID': 'exp001',
        '별칭': 'baseline',
        '주요 변경사항': 'PPO 기본 구현 (lr=3e-4, n_steps=2048, 1M steps)',
        'best Sharpe': 0.795,
        'best Step': '100k',
        '문제점': '100k 이후 Sharpe 0.15~0.35로 하락, 수렴 실패'
    },
    {
        'ID': 'exp002',
        '별칭': 'tuned',
        '주요 변경사항': 'lr 감소 (3e-4 → 1e-4), n_steps=4096',
        'best Sharpe': 0.745,
        'best Step': '50k',
        '문제점': '850k 이후 Sharpe 음수, 후반 불안정'
    },
    {
        'ID': 'exp003',
        '별칭': 'vecnorm_cosine',
        '주요 변경사항': 'VecNormalize + cosine LR 스케줄',
        'best Sharpe': '— (버그)',
        'best Step': '—',
        '문제점': '수수료 이중 계산 버그 → 0 거래 에피소드, 결과 무효'
    },
    {
        'ID': 'exp004',
        '별칭': 'atr_scaled_action',
        '주요 변경사항': '버그 수정 + ATR 비례 action 공식 도입, n_steps=8192',
        'best Sharpe': 4.588,
        'best Step': '100k',
        '문제점': '최소 gap이 너무 작아(0.1×ATR) 수수료 < gap, 불안정'
    },
    {
        'ID': 'exp005',
        '별칭': 'min_gap_fix',
        '주요 변경사항': '최소 gap 계수 상향: 0.1→0.5×ATR (round-trip +0.25% 보장)',
        'best Sharpe': 1.060,
        'best Step': '100k',
        '문제점': '후반에 0 거래 수렴 (공통 문제), Return≈0.00%'
    },
]

hist_df = pd.DataFrame(history).set_index('ID')
print('=== 실험 히스토리 요약 ===')
print(hist_df.to_string())

## 2. exp001 / exp002 학습 곡선 파싱

train_log.txt에서 `[eval]` 행을 추출하여 DataFrame으로 변환한다.

In [ ]:
EVAL_PATTERN = r'\[eval\] step=\s*([\d,]+) \| Sharpe=([+-]?\d+\.\d+) \| Return=([+-]?\d+\.\d+)% \| MDD=(\d+\.\d+)%'

def parse_eval_log(path: str) -> pd.DataFrame:
    """로그 파일에서 [eval] 행을 파싱해 DataFrame으로 반환한다."""
    with open(path, encoding='utf-8', errors='replace') as f:
        log = f.read()
    matches = re.findall(EVAL_PATTERN, log)
    df = pd.DataFrame(matches, columns=['step', 'sharpe', 'return_pct', 'mdd_pct'])
    df['step']       = df['step'].str.replace(',', '').astype(int)
    df['sharpe']     = df['sharpe'].astype(float)
    df['return_pct'] = df['return_pct'].astype(float)
    df['mdd_pct']    = df['mdd_pct'].astype(float)
    return df

# exp001, exp002 파싱
df_exp001 = parse_eval_log('../experiments/exp001_baseline/train_log.txt')
df_exp002 = parse_eval_log('../experiments/exp002_tuned/train_log.txt')

# exp005 파싱 (eval_log.txt 사용)
df_exp005 = parse_eval_log('../experiments/exp005_min_gap_fix/eval_log.txt')

print(f'exp001: {len(df_exp001)}개 평가점  | best Sharpe={df_exp001["sharpe"].max():.3f} (step={df_exp001.loc[df_exp001["sharpe"].idxmax(), "step"]:,})')
print(f'exp002: {len(df_exp002)}개 평가점  | best Sharpe={df_exp002["sharpe"].max():.3f} (step={df_exp002.loc[df_exp002["sharpe"].idxmax(), "step"]:,})')
print(f'exp005: {len(df_exp005)}개 평가점  | best Sharpe={df_exp005["sharpe"].max():.3f} (step={df_exp005.loc[df_exp005["sharpe"].idxmax(), "step"]:,})')

## 3. exp001 / exp002 학습 곡선 (Sharpe / Return / MDD)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('exp001 vs exp002 학습 곡선 (Val 셋)', fontsize=13, fontweight='bold')

def plot_curve(ax, df, label, color, metric, ylabel, title):
    ax.plot(df['step'], df[metric], '-o', ms=4, lw=1.5, color=color, label=label)
    best_idx = df[metric].idxmax() if metric == 'sharpe' else df[metric].idxmin()
    ax.scatter([df.loc[best_idx, 'step']], [df.loc[best_idx, metric]],
               color=color, s=120, zorder=5, marker='*')

# Sharpe
ax = axes[0]
plot_curve(ax, df_exp001, f'exp001 (best={df_exp001["sharpe"].max():.3f})', '#1565C0', 'sharpe', 'Sharpe', 'Val Sharpe Ratio')
plot_curve(ax, df_exp002, f'exp002 (best={df_exp002["sharpe"].max():.3f})', '#E53935', 'sharpe', 'Sharpe', 'Val Sharpe Ratio')
ax.axhline(0, color='black', lw=0.7, ls=':', alpha=0.5)
ax.set_title('Val Sharpe Ratio'); ax.set_xlabel('학습 스텝'); ax.set_ylabel('Sharpe')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# Return
ax = axes[1]
ax.plot(df_exp001['step'], df_exp001['return_pct'], '-o', ms=4, lw=1.5, color='#1565C0', label='exp001')
ax.plot(df_exp002['step'], df_exp002['return_pct'], '-o', ms=4, lw=1.5, color='#E53935', label='exp002')
ax.axhline(0, color='black', lw=0.7, ls=':', alpha=0.5)
ax.set_title('Val 누적 수익률 (%)'); ax.set_xlabel('학습 스텝'); ax.set_ylabel('수익률 (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# MDD
ax = axes[2]
ax.plot(df_exp001['step'], df_exp001['mdd_pct'], '-o', ms=4, lw=1.5, color='#1565C0', label='exp001')
ax.plot(df_exp002['step'], df_exp002['mdd_pct'], '-o', ms=4, lw=1.5, color='#E53935', label='exp002')
ax.set_title('Val MDD (%)'); ax.set_xlabel('학습 스텝'); ax.set_ylabel('MDD (%)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('../reports/semester1/figures/03_exp0102_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 03_exp0102_learning_curves.png')

## 4. exp005 학습 곡선 (3M steps)

exp005의 특징: Return≈0.00%이지만 초반 Sharpe가 1.060까지 상승한 뒤 하락.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('exp005 학습 곡선 (min_gap_fix, 3M steps)', fontsize=13, fontweight='bold')

best_idx = df_exp005['sharpe'].idxmax()
best_step = df_exp005.loc[best_idx, 'step']
best_sharpe = df_exp005.loc[best_idx, 'sharpe']

# Sharpe
ax = axes[0]
ax.plot(df_exp005['step'], df_exp005['sharpe'], '-', lw=1.5, color='#6A1B9A', alpha=0.8, label='Val Sharpe')
ax.scatter([best_step], [best_sharpe], color='gold', s=180, zorder=5, marker='*',
           label=f'Best: {best_sharpe:.3f} (step={best_step/1e3:.0f}k)')
ax.axhline(0, color='black', lw=0.7, ls=':', alpha=0.5)
ax.set_title('Val Sharpe Ratio'); ax.set_xlabel('학습 스텝'); ax.set_ylabel('Sharpe')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
# 후반 0 거래 구간 음영
cutoff = 1_000_000
ax.axvspan(cutoff, df_exp005['step'].max(), alpha=0.08, color='red')
ax.text(cutoff * 1.02, best_sharpe * 0.85, '0거래 수렴', color='red', fontsize=9, alpha=0.7)

# MDD
ax = axes[1]
ax.plot(df_exp005['step'], df_exp005['mdd_pct'], '-', lw=1.5, color='#C62828', alpha=0.8)
ax.set_title('Val MDD (%)'); ax.set_xlabel('학습 스텝'); ax.set_ylabel('MDD (%)')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('../reports/semester1/figures/03_exp005_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 03_exp005_learning_curve.png')

## 5. exp001 / exp002 / exp005 Sharpe 학습 곡선 오버레이

exp003/exp004는 버그 실험이므로 생략 (표에서만 언급).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# 3개 실험 오버레이
exp_specs = [
    (df_exp001, 'exp001 (lr=3e-4, 1M)', '#1565C0', '-'),
    (df_exp002, 'exp002 (lr=1e-4, 1M)', '#E53935', '-'),
    (df_exp005, 'exp005 (min_gap_fix, 3M)', '#6A1B9A', '-'),
]

for df, label, color, ls in exp_specs:
    ax.plot(df['step'], df['sharpe'], ls=ls, lw=1.8, color=color, label=label, alpha=0.85)
    best_idx = df['sharpe'].idxmax()
    ax.scatter([df.loc[best_idx, 'step']], [df.loc[best_idx, 'sharpe']],
               color=color, s=140, zorder=5, marker='*')

ax.axhline(0, color='black', lw=0.8, ls=':', alpha=0.5, label='Sharpe=0')
ax.axhline(1.0, color='gray', lw=0.8, ls='--', alpha=0.4, label='Sharpe=1.0')

ax.set_title('실험별 Val Sharpe 학습 곡선 비교 (exp001 / 002 / 005)', fontsize=13, fontweight='bold')
ax.set_xlabel('학습 스텝'); ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# 공통 패턴 주석: 초반 학습, 후반 하락
ax.annotate('초반 학습\n(모든 실험 공통)', xy=(100_000, 0.9), xytext=(250_000, 0.7),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
            fontsize=9, color='black')

plt.tight_layout()
plt.savefig('../reports/semester1/figures/03_learning_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 03_learning_curves_comparison.png')

## 6. 핵심 문제 진단: 에피소드 길이 vs Discount 효과

**근본 원인**: 에피소드가 25,916스텝(Train 3년)으로 너무 길다.

γ=0.99에서 스텝 t의 reward가 에이전트에 미치는 영향: `γ^t`

- t=2,000에서 `0.99^2000 ≈ 2e-9`
- 즉, 에피소드 후반 거의 모든 reward가 **사실상 0으로 무시된다**

In [ ]:
gamma = 0.99
steps = np.arange(0, 25_917)
discount_effect = gamma ** steps

# γ^t < 1e-9 구간
negligible_threshold = 1e-9
cutoff_step = np.argmax(discount_effect < negligible_threshold)
print(f'γ^{cutoff_step} = {discount_effect[cutoff_step]:.2e}  ← 이 이후 reward 무시')
print(f'전체 에피소드: 25,916 스텝')
print(f'유효 구간: 0 ~ {cutoff_step} 스텝 ({cutoff_step/25916*100:.1f}%)')

fig, ax = plt.subplots(figsize=(12, 5))

# 유효 구간 (파랑)
valid_steps = steps[:cutoff_step+1]
ax.semilogy(valid_steps, discount_effect[:cutoff_step+1],
            color='#1565C0', lw=2, label=f'유효 구간 (0~{cutoff_step}step)')

# 무시 구간 (빨강 음영)
ax.semilogy(steps[cutoff_step:], discount_effect[cutoff_step:],
            color='#E53935', lw=1, alpha=0.4)
ax.axvspan(cutoff_step, 25_916, alpha=0.08, color='red',
           label=f'Reward 무시 구간 ({cutoff_step}~25,916step, {(25916-cutoff_step)/25916*100:.0f}%)')

# 주석
ax.axhline(negligible_threshold, color='gray', ls='--', lw=1.2, alpha=0.7)
ax.text(cutoff_step + 200, negligible_threshold * 5,
        f'γ^{cutoff_step} ≈ {negligible_threshold:.0e}\n(이 이후 reward\n사실상 0으로 무시)', fontsize=9, color='#E53935')

ax.set_xlabel('에피소드 내 스텝 (t)', fontsize=11)
ax.set_ylabel('discount 효과 γ^t  (log scale)', fontsize=11)
ax.set_title(f'에피소드 길이 vs Discount 효과 (γ={gamma}, 에피소드=25,916 step)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

# 실제 에피소드 끝 표시
ax.axvline(25_916, color='purple', ls=':', lw=1.5, alpha=0.7)
ax.text(25_916 - 1500, 1e-3, 'Episode end\n(25,916)', fontsize=8, color='purple')

plt.tight_layout()
plt.savefig('../reports/semester1/figures/03_discount_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 03_discount_effect.png')
print(f'\n결론: 전체 에피소드의 {(25916-cutoff_step)/25916*100:.0f}%에 해당하는 reward가 학습에 기여하지 않음!')

## 7. 실험별 성능 비교 표

베이스라인 포함 전체 비교.

> **주의**: exp003/exp004는 버그/불안정으로 공식 결과에서 제외

In [ ]:
# 베이스라인은 노트북 02에서 얻은 수치 (Val 셋)
comparison_data = [
    {'모델': 'Fixed Grid 1%',         '수익률(%)': 43.16,  'Sharpe': 2.610, 'MDD(%)':  10.77, '거래수': 567,  '비고': '베이스라인'},
    {'모델': 'Buy-and-Hold',           '수익률(%)': 150.18, 'Sharpe': 2.377, 'MDD(%)':  21.74, '거래수': 1,    '비고': '베이스라인'},
    {'모델': 'Fixed Grid 2%',         '수익률(%)': 16.96,  'Sharpe': 2.032, 'MDD(%)':   7.65, '거래수': 126,  '비고': '베이스라인'},
    {'모델': 'ATR Grid k=2.0',        '수익률(%)': 28.98,  'Sharpe': 1.948, 'MDD(%)':   9.38, '거래수': 320,  '비고': '베이스라인'},
    {'모델': 'ATR Grid k=1.0',        '수익률(%)': 39.79,  'Sharpe': 1.434, 'MDD(%)':  15.86, '거래수': 833,  '비고': '베이스라인'},
    {'모델': 'Fixed Grid 5%',         '수익률(%)':  2.47,  'Sharpe': 1.375, 'MDD(%)':   1.67, '거래수': 8,    '비고': '베이스라인'},
    {'모델': 'exp005 best (100k)',     '수익률(%)':  0.00,  'Sharpe': 1.060, 'MDD(%)':  81.12, '거래수': '-',  '비고': 'PPO (최신)'},
    {'모델': 'exp001 best (100k)',     '수익률(%)':  1.20,  'Sharpe': 0.795, 'MDD(%)':   0.82, '거래수': 8,    '비고': 'PPO'},
    {'모델': 'exp002 best (50k)',      '수익률(%)':  7.64,  'Sharpe': 0.745, 'MDD(%)':  11.43, '거래수': '-',  '비고': 'PPO'},
    {'모델': 'exp003',                 '수익률(%)': '-',    'Sharpe': '-',   'MDD(%)':  '-',   '거래수': 0,    '비고': '버그 (수수료 이중계산)'},
    {'모델': 'exp004 best (100k)',     '수익률(%)':  0.00,  'Sharpe': 4.588, 'MDD(%)':  '-',   '거래수': '-',  '비고': 'PPO (불안정, 제외)'},
]

comp_df = pd.DataFrame(comparison_data)
print('=== 전체 실험 성능 비교 (Val 셋, 2023) ===')
print(comp_df.to_string(index=False))

## 8. 결론 및 exp006 예고

### 공통 문제: 0 거래 수렴

모든 실험(exp001~005)에서 공통으로 관찰된 패턴:
1. **초반** (0~150k steps): Sharpe 상승, 정상 거래
2. **후반** (150k~ steps): 점진적으로 0 거래로 수렴, Sharpe 하락

### 근본 원인 (진단 완료)

| 원인 | 설명 |
|------|------|
| **에피소드 과다 길이** | Train 에피소드 = 25,916 스텝 (3년 hourly) |
| **Discount 붕괴** | γ=0.99에서 t≥2,000 이후 γ^t ≈ 0 |
| **Policy 편향** | 후반 reward가 0 → 에이전트가 "거래 안 함"을 최적 정책으로 학습 |

### exp006 설계 (예고)

- **에피소드 단축**: `max_episode_steps = 2000` (약 83일)
  - γ^2000 ≈ 2e-9이므로, 2000스텝이면 discount 붕괴 전에 에피소드 종료
  - 동일 데이터로 더 많은 에피소드 → 샘플 효율성 향상
- **멀티 환경 병렬화**: `n_envs = 8` (SubprocVecEnv)
  - 더 다양한 시장 조건 동시 학습
- **n_steps 조정**: 2048 (단일 에피소드 커버 가능한 크기로)

---

## 9. exp006 / exp007 / exp008 학습 곡선 (신규 실험)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ── exp006 파싱 ─────────────────────────────────────────────
EVAL_PATTERN_SHORT = r'\[eval\] step=\s*([\d,]+) \| Sharpe=([+-]?\d+\.\d+)'

def parse_eval_log_short(path: str) -> pd.DataFrame:
    """Sharpe만 파싱 (Return/MDD 없는 로그 포맷 대응)."""
    with open(path, encoding='utf-8', errors='replace') as f:
        log = f.read()
    matches = re.findall(EVAL_PATTERN_SHORT, log)
    df = pd.DataFrame(matches, columns=['step', 'sharpe'])
    df['step']   = df['step'].str.replace(',', '').astype(int)
    df['sharpe'] = df['sharpe'].astype(float)
    return df

df_exp006 = parse_eval_log_short('../experiments/exp006_short_episode_multienv/eval_log.txt')

# ── exp007: 로그 없음 → 고정 포인트만 ───────────────────────
# train_log 미저장. 평가 결과(버그 수정 후)만 알려짐.
exp007_known = {'step': 'N/A', 'sharpe': 11.884, 'return_pct': 56.1, 'mdd_pct': 2.89}

# ── exp008 파싱 ─────────────────────────────────────────────
df_exp008 = parse_eval_log('../experiments/exp008_ent_coef_05/train_log.txt')

print(f'exp006: {len(df_exp006)}개 평가점 | best Sharpe={df_exp006["sharpe"].max():.3f} '
      f'(step={df_exp006.loc[df_exp006["sharpe"].idxmax(), "step"]:,})')
print(f'exp007: 로그 없음 | 알려진 best Sharpe={exp007_known["sharpe"]:.3f} (Return={exp007_known["return_pct"]}%, MDD={exp007_known["mdd_pct"]}%)')
print(f'exp008: {len(df_exp008)}개 평가점 | best Sharpe={df_exp008["sharpe"].max():.3f} '
      f'(step={df_exp008.loc[df_exp008["sharpe"].idxmax(), "step"]:,})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('exp006 / exp007 / exp008 학습 곡선 (Val 셋)', fontsize=13, fontweight='bold')

# ── exp006 ──────────────────────────────────────────────────
ax = axes[0]
ax.plot(df_exp006['step'], df_exp006['sharpe'], '-o', ms=3, lw=1.5, color='#0277BD', alpha=0.85)
best_idx006 = df_exp006['sharpe'].idxmax()
best_s006   = df_exp006.loc[best_idx006, 'sharpe']
best_t006   = df_exp006.loc[best_idx006, 'step']
ax.scatter([best_t006], [best_s006], color='gold', s=180, zorder=5, marker='*',
           label=f'Best: {best_s006:.3f} (step={best_t006/1e3:.0f}k)')
ax.axhline(0, color='black', lw=0.7, ls=':', alpha=0.5)
ax.axhline(1.0, color='gray', lw=0.8, ls='--', alpha=0.4)
ax.set_title('exp006: ep단축(2016) + n_envs=4 + VecNorm', fontsize=10)
ax.set_xlabel('학습 스텝'); ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# ── exp007 (로그 없음) ───────────────────────────────────────
ax = axes[1]
ax.text(0.5, 0.6, '학습 중 로그 미저장\n(train_log 없음)',
        ha='center', va='center', fontsize=11, color='#555555',
        transform=ax.transAxes)
ax.text(0.5, 0.38,
        f'평가 결과 (best_model, 버그수정 후)\n'
        f'Sharpe = {exp007_known["sharpe"]:.3f}\n'
        f'Return = +{exp007_known["return_pct"]}%\n'
        f'MDD    = {exp007_known["mdd_pct"]}%',
        ha='center', va='center', fontsize=10, color='#1A237E',
        transform=ax.transAxes,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8EAF6', edgecolor='#3F51B5', lw=1.2))
ax.set_title('exp007: VecNorm 제거 + multi-episode 평가(5회)', fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')

# ── exp008 ──────────────────────────────────────────────────
ax = axes[2]
ax.plot(df_exp008['step'], df_exp008['sharpe'], '-', lw=1.5, color='#2E7D32', alpha=0.85)
best_idx008 = df_exp008['sharpe'].idxmax()
best_s008   = df_exp008.loc[best_idx008, 'sharpe']
best_t008   = df_exp008.loc[best_idx008, 'step']
ax.scatter([best_t008], [best_s008], color='gold', s=180, zorder=5, marker='*',
           label=f'Best: {best_s008:.3f} (step={best_t008/1e3:.0f}k)')
# 레짐 안정 구간 표시 (700k~1700k)
ax.axvspan(700_000, 1_700_000, alpha=0.07, color='green')
ax.text(750_000, best_s008 * 0.97, '레짐 적응\n안정 구간', fontsize=8, color='darkgreen', alpha=0.8)
ax.axhline(0, color='black', lw=0.7, ls=':', alpha=0.5)
ax.set_title('exp008: ent_coef 0.01 → 0.05', fontsize=10)
ax.set_xlabel('학습 스텝'); ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('../reports/semester1/figures/03_exp006_007_008_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 03_exp006_007_008_curves.png')

## 10. 전체 실험 히스토리 비교표 (exp001 ~ exp008 + Baseline)

In [ ]:
full_history = [
    {
        '실험명':    'Fixed Grid 1%',
        '핵심 변경': '베이스라인 (고정 그리드)',
        'best Sharpe': 2.610,
        'best Step': '—',
        '비고': '베이스라인 기준선',
    },
    {
        '실험명':    'exp001',
        '핵심 변경': 'PPO 기본 (lr=3e-4, n_steps=2048, 1M)',
        'best Sharpe': 0.795,
        'best Step': '100k',
        '비고': '100k 이후 Sharpe 하락, 수렴 실패',
    },
    {
        '실험명':    'exp002',
        '핵심 변경': 'lr↓(3e-4→1e-4) + n_steps=4096',
        'best Sharpe': 0.745,
        'best Step': '50k',
        '비고': '850k 이후 Sharpe 음수, 불안정',
    },
    {
        '실험명':    'exp003',
        '핵심 변경': 'VecNormalize + cosine LR 스케줄',
        'best Sharpe': '— (버그)',
        'best Step': '—',
        '비고': '수수료 이중계산 버그 → 결과 무효',
    },
    {
        '실험명':    'exp004',
        '핵심 변경': '버그수정 + ATR 비례 action 공식 도입',
        'best Sharpe': 4.588,
        'best Step': '100k',
        '비고': 'eval 버그로 신뢰 불가 (제외)',
    },
    {
        '실험명':    'exp005',
        '핵심 변경': '최소 gap 0.1→0.5×ATR',
        'best Sharpe': 1.060,
        'best Step': '100k',
        '비고': '후반 0거래 수렴 (Return≈0%), numpy 비호환 이슈',
    },
    {
        '실험명':    'exp006',
        '핵심 변경': '에피소드 단축(2016) + n_envs=4 + VecNorm 활성',
        'best Sharpe': 1.183,
        'best Step': '550k',
        '비고': 'VecNorm 상태 정규화 활성 → 스케일 혼재 문제',
    },
    {
        '실험명':    'exp007',
        '핵심 변경': 'VecNorm 비활성 + multi-episode 평가(5회)',
        'best Sharpe': 11.884,
        'best Step': '— (로그 없음)',
        '비고': 'eval 버그 수정 후 수치. 학습 중 로그 미저장',
    },
    {
        '실험명':    'exp008',
        '핵심 변경': 'ent_coef 0.01→0.05 (탐색 강화)',
        'best Sharpe': 12.381,
        'best Step': '850k',
        '비고': 'Return=+57.7%, MDD=2.90%. 레짐 적응 확인 ★',
    },
]

hist_full_df = pd.DataFrame(full_history)

# 스타일 적용 (best Sharpe 기준 하이라이트)
def highlight_best(row):
    try:
        v = float(row['best Sharpe'])
    except (ValueError, TypeError):
        return [''] * len(row)
    styles = []
    for col in row.index:
        if col == 'best Sharpe' and v >= 10:
            styles.append('background-color: #C8E6C9; font-weight: bold')
        elif col == 'best Sharpe' and v >= 2:
            styles.append('background-color: #FFF9C4')
        elif '비고' in col and '버그' in str(row['비고']):
            styles.append('color: #B71C1C')
        else:
            styles.append('')
    return styles

display(hist_full_df.style.apply(highlight_best, axis=1).set_caption('전체 실험 히스토리 비교 (Val 셋, 2023)'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('전체 실험 학습 곡선 오버레이 (exp001 ~ exp008)', fontsize=13, fontweight='bold')

# ── 왼쪽: exp001 / exp002 / exp005 / exp006 (Sharpe < 2) ─────
ax = axes[0]
early_specs = [
    (df_exp001, 'exp001 (best=0.795)', '#1565C0', '-'),
    (df_exp002, 'exp002 (best=0.745)', '#E53935', '-'),
    (df_exp005, 'exp005 (best=1.060)', '#6A1B9A', '-'),
    (df_exp006, 'exp006 (best=1.183)', '#0277BD', '--'),
]
for df, label, color, ls in early_specs:
    ax.plot(df['step'], df['sharpe'], ls=ls, lw=1.5, color=color, label=label, alpha=0.85)
    best_idx = df['sharpe'].idxmax()
    ax.scatter([df.loc[best_idx, 'step']], [df.loc[best_idx, 'sharpe']],
               color=color, s=100, zorder=5, marker='*')
ax.axhline(0, color='black', lw=0.7, ls=':', alpha=0.5)
ax.axhline(1.0, color='gray', lw=0.8, ls='--', alpha=0.4, label='Sharpe=1.0')
ax.set_title('초기 실험 (exp001~006)', fontsize=11)
ax.set_xlabel('학습 스텝'); ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# ── 오른쪽: exp008 vs Fixed Grid 1% 기준선 ───────────────────
ax = axes[1]
ax.plot(df_exp008['step'], df_exp008['sharpe'], '-', lw=2.0, color='#2E7D32',
        alpha=0.9, label=f'exp008 (best={df_exp008["sharpe"].max():.3f})')
best_idx = df_exp008['sharpe'].idxmax()
ax.scatter([df_exp008.loc[best_idx, 'step']], [df_exp008.loc[best_idx, 'sharpe']],
           color='gold', s=180, zorder=5, marker='*')

# exp007 포인트 (x축 없음 → 전 구간 수평선으로 표시)
ax.axhline(exp007_known['sharpe'], color='#1565C0', lw=1.5, ls='--', alpha=0.7,
           label=f'exp007 best={exp007_known["sharpe"]:.3f} (로그없음, 수평선)')

# 베이스라인 기준선
ax.axhline(2.610, color='#E65100', lw=1.5, ls=':', alpha=0.8, label='Fixed Grid 1% (Sharpe=2.610)')

ax.set_title('exp007 / exp008 vs 베이스라인 비교', fontsize=11)
ax.set_xlabel('학습 스텝'); ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('../reports/semester1/figures/03_learning_curve_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: reports/semester1/figures/03_learning_curve_all.png')